In [1]:
import pandas as pd

In [2]:
sample_df = pd.read_csv("sample_circuits.csv")
master_df = pd.read_csv("../../master.csv")

In [3]:
single_opt = master_df[
        (master_df["opt_2"].isna())
        & (master_df["opt_3"].isna())
        ].reset_index()

In [14]:
sample_one_opt= single_opt[single_opt["original_circuit_path"].isin(sample_df["qasm_path"])]

sample_one_opt["original_circuit"].value_counts()

original_circuit
efficient_su2_12                  31
efficient_su2_8                   28
qaoa_barabasi_albert_N10_3reps    13
qaoa_barabasi_albert_N11_3reps    13
qft_N007                          13
qft_N013                          13
qv_N002_12345                     13
qv_N007_12345                     12
real_amplitudes_12_r3             12
square_heisenberg_N4              12
tof_10                            11
tof_5                             11
real_amplitudes_8_r2              11
square_heisenberg_N16             11
Name: count, dtype: int64

In [16]:
best_one_opt = (
    sample_one_opt
    .sort_values(
        by=[
            "original_circuit",
            "opt_1",
            "final_two_qubit_gates",
            "final_total_gates"
        ],
        ascending=[True, True, True, True]
    )
    .drop_duplicates(
        subset=["original_circuit", "opt_1"],
        keep="first"
    )
    .reset_index(drop=True)
)
best_one_opt

,index,id,parent_circuit_id,chain_name,original_circuit,original_category,original_circuit_path,opt_1,opt_2,opt_3,qasm_path,initial_num_qubits,initial_depth,initial_two_qubit_gates,initial_total_gates,final_num_qubits,final_depth,final_two_qubit_gates,final_total_gates,opt_chain
0,51,52,2,single_4,efficient_su2_12,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_12....,qiskit_ai,NaN,NaN,NaN,12,25,66,114,0,73,115,163,efficient_su2_12__qiskit_ai
1,55,56,2,single_5,efficient_su2_12,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_12....,qiskit_standard,NaN,NaN,NaN,12,25,66,114,0,134,171,406,efficient_su2_12__qiskit_standard
2,35,36,2,single_3,efficient_su2_12,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_12....,tket,NaN,NaN,NaN,12,25,66,114,0,29,66,162,efficient_su2_12__tket
3,33,34,2,single_2,efficient_su2_12,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_12....,wisq_bqskit,NaN,NaN,NaN,12,25,66,114,0,31,11,107,efficient_su2_12__wisq_bqskit
4,31,32,2,single_1,efficient_su2_12,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_12....,wisq_rules,NaN,NaN,NaN,12,25,66,114,0,36,11,107,efficient_su2_12__wisq_rules
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,626,627,42,single_4,tof_5,feynman,benchmarks/ai_transpile/qasm/feynman/tof_5.qasm,qiskit_ai,NaN,NaN,NaN,9,23,0,35,0,8,0,28,tof_5__qiskit_ai
66,628,629,42,single_5,tof_5,feynman,benchmarks/ai_transpile/qasm/feynman/tof_5.qasm,qiskit_standard,NaN,NaN,NaN,9,23,0,35,0,137,58,206,tof_5__qiskit_standard
67,623,624,42,single_3,tof_5,feynman,benchmarks/ai_transpile/qasm/feynman/tof_5.qasm,tket,NaN,NaN,NaN,9,23,0,35,0,82,42,120,tof_5__tket
68,622,623,42,single_2,tof_5,feynman,benchmarks/ai_transpile/qasm/feynman/tof_5.qasm,wisq_bqskit,NaN,NaN,NaN,9,23,0,35,0,124,32,222,tof_5__wisq_bqskit


In [19]:
def is_exhaustive(df, optimizers):
    optimizers_tested = set(df["opt_1"].dropna().unique().tolist())
    if optimizers == optimizers_tested:
        return -1
    return list(set(optimizers) - set(optimizers_tested))

    

In [25]:
optimizers = set(["wisq_rules", "wisq_bqskit", "tket", "qiskit_ai", "qiskit_standard"])
sample_circuits = best_one_opt["original_circuit"].dropna().unique().tolist()
df = best_one_opt
for circuit in sample_circuits:
    temp_df = df[
        (df["original_circuit"] == circuit)
        & (df["opt_2"].isna())
        & (df["opt_3"].isna())
        ]    
    exhaustive = is_exhaustive(temp_df, optimizers)
    if exhaustive != -1:
        print(f"{circuit} has not been tested exhaustively.")
        print(f"Untested optimizers: {exhaustive}\n")
    else:
        print(f"{circuit} has been tested exhaustively.\n")
    



efficient_su2_12 has been tested exhaustively.

efficient_su2_8 has been tested exhaustively.

qaoa_barabasi_albert_N10_3reps has been tested exhaustively.

qaoa_barabasi_albert_N11_3reps has been tested exhaustively.

qft_N007 has been tested exhaustively.

qft_N013 has been tested exhaustively.

qv_N002_12345 has been tested exhaustively.

qv_N007_12345 has been tested exhaustively.

real_amplitudes_12_r3 has been tested exhaustively.

real_amplitudes_8_r2 has been tested exhaustively.

square_heisenberg_N16 has been tested exhaustively.

square_heisenberg_N4 has been tested exhaustively.

tof_10 has been tested exhaustively.

tof_5 has been tested exhaustively.



In [24]:
best_one_opt.to_csv("best_single_opt.csv", index=False)

In [26]:
print(sample_circuits)

['efficient_su2_12', 'efficient_su2_8', 'qaoa_barabasi_albert_N10_3reps', 'qaoa_barabasi_albert_N11_3reps', 'qft_N007', 'qft_N013', 'qv_N002_12345', 'qv_N007_12345', 'real_amplitudes_12_r3', 'real_amplitudes_8_r2', 'square_heisenberg_N16', 'square_heisenberg_N4', 'tof_10', 'tof_5']
